In [1]:
print("Hello WOrld")

Hello WOrld


## Qdrant VectorStore retriver


In [92]:
from qdrant_client import QdrantClient, models
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client.http.models import Distance, SparseVectorParams, VectorParams
from helpers.common import ollama_llm, langfuse_handler

QDRANT_BASE_URL = "http://localhost:6333"
QDRANT_COLLECTION_NAME = "langchain_ollama"

OLLAMA_EMBEDDING_MODEL = "nomic-embed-text:latest"
OLLAMA_BASE_URL = "http://localhost:11434/"

## Embeddings definition

In [65]:
dense_embeddings = OllamaEmbeddings(
    model=OLLAMA_EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL
)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")


In [66]:
client = QdrantClient(
      url=QDRANT_BASE_URL
)
vector_store = QdrantVectorStore(
    client=client,
    collection_name= QDRANT_COLLECTION_NAME,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse"  
)

In [67]:
client.get_collection(QDRANT_COLLECTION_NAME)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=311, points_count=311, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=False, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_si

In [68]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
retrieved_docs = retriever.invoke("How to gain muscle mass")

In [78]:
def format_docs(docs):
  return "\n\n".join([doc.page_content for doc in docs])

# format_docs(retrieved_docs)

## Define the Prompt

In [ ]:
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

human_message = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.
Answer in 3 bullet points and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

human_prompt = HumanMessagePromptTemplate.from_template(human_message)
messages = [human_prompt]
chat_prompt = ChatPromptTemplate(messages)

# chat_prompt.invoke({
#   'context': "Some Context",
#   "question": "WHy?"
# })

In [91]:
rag_chain = (
  {"context": retriever | format_docs, "question": RunnablePassthrough()}
  | chat_prompt
  | ollama_llm
  | StrOutputParser()
)

rag_chain.invoke('How to gain muscle mass?')

'Here’s a concise answer to your question, based on the provided context:\n\n*   Creatine monohydrate is a well-known supplement used to gain muscle mass and support performance and recovery.\n*   Respondents consume fitness supplements with multiple goals including increasing muscle mass, enhancing performance, and meeting nutritional needs.\n*   A positive correlation was observed between training frequency and supplement usage, although the correlation is weak.'

In [ ]:
for message in rag_chain.stream("How to gain muscle mass?", 
config={"callbacks": [langfuse_handler], "run_name": "rag-chat"},
):
  print(message, end="")

Here’s a concise answer to your question, based on the provided context:

*   Creatine monohydrate is a well-known supplement used to gain muscle mass and support performance and recovery.
*   Respondents consume fitness supplements to increase muscle mass (49%), improve health (47%), and improve sport-specific performance (28%).
*   Supplement use is often linked to training frequency, though the correlation is weak and doesn't predict individual supplement intake.

#### Can pull the prompt from langsmith as well

In [ ]:
# from langsmith import Client

# client = Client()
# prompt = client.pull_prompt("rlm/rag-prompt")

/home/yash/Desktop/Code/1. Langchain and Ollama/lang_ollama_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
